In [1]:
import sys
from tqdm import tqdm
sys.path.append("..")


import matplotlib.pyplot as plt
import numpy as np
import torch

DATA_DIR = "../mu3e_trigger_data"
MODEL_DIR = "../models"
PLOTS_DIR = "../plots"
from src.data_preparation import load_numpy_files

signal_prefix = f"{DATA_DIR}/sig"
background_prefix = f"{DATA_DIR}/bg"
signal_only_prefix = f"{DATA_DIR}/sig_only"


In [2]:
import src.torch.pre_processing.graph_batching as graph_batching
from importlib import reload
reload(graph_batching)

<module 'src.torch.pre_processing.graph_batching' from '/Users/simi/mu3e_trigger/notebooks_supervised/../src/torch/pre_processing/graph_batching.py'>

In [3]:
train_dataset, test_dataset = graph_batching.create_dataset(
    signal_prefix,
    has_layer_feature=True,
    n_events=30000,
    split=(0.7, 0.3),
    type="hetero",
    whole_event_mode=False,
    timing_cutoff=8,
    mppc_timing_cutoff=2,)


In [4]:
from torch_geometric.loader import DataLoader
train_loader = DataLoader(train_dataset, batch_size=512, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=512, shuffle=False)

In [5]:
import torch
from torch import nn
from torch_geometric.data import HeteroData
from torch_geometric.nn import HeteroConv, SAGEConv, BatchNorm


def get_mlp(in_dim, out_dim, hidden_dim=64, num_layers=2):
    layers = []
    dim = in_dim
    for _ in range(num_layers - 1):
        layers.append(nn.Linear(dim, hidden_dim))
        layers.append(nn.ReLU())
        dim = hidden_dim
    layers.append(nn.Linear(dim, out_dim))
    return nn.Sequential(*layers)


class TrackingHeteroGNN(nn.Module):
    def __init__(self, hidden_dim=64, num_layers=3, dropout=0.1, use_rel_feats=True):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers
        self.dropout = dropout
        self.use_rel_feats = use_rel_feats

        # Node encoders
        self.encoders = nn.ModuleDict({
            "pixel": get_mlp(4, hidden_dim, hidden_dim),
            "mppc": get_mlp(4, hidden_dim, hidden_dim),  # [x, y, z, time]
        })

        # Message passing layers
        self.convs = nn.ModuleList()
        self.norms = nn.ModuleList()
        for _ in range(num_layers):
            conv = HeteroConv({
                edge_type: SAGEConv((hidden_dim, hidden_dim), hidden_dim)
                for edge_type in [
                    ("pixel", "to", "mppc"),
                    ("mppc", "to", "pixel"),
                    ("pixel", "to", "pixel"),
                    ("mppc", "to", "mppc"),
                ]
            }, aggr="mean")
            self.convs.append(conv)

            norm = nn.ModuleDict({
                "pixel": BatchNorm(hidden_dim),
                "mppc": BatchNorm(hidden_dim),
            })
            self.norms.append(norm)

        # Edge classifiers per edge type
        edge_in_dim = 2 * hidden_dim + (4 if use_rel_feats else 0)
        self.edge_classifiers = nn.ModuleDict({
            "__".join(edge_type): get_mlp(edge_in_dim, 1, hidden_dim, num_layers=3)
            for edge_type in [
                ("pixel", "to", "mppc"),
                ("mppc", "to", "pixel"),
                ("pixel", "to", "pixel"),
                ("mppc", "to", "mppc"),
            ]
        })

    def forward(self, data: HeteroData):
        x_dict, edge_index_dict = data.x_dict, data.edge_index_dict

        # Normalize time feature for MPPC (map [0,8] -> [0,1])
        if "mppc" in x_dict:
            x_dict["mppc"] = x_dict["mppc"].clone()
            x_dict["mppc"][:, -1] = x_dict["mppc"][:, -1] / 8.0

        # Initial encodings
        for node_type, encoder in self.encoders.items():
            if node_type in x_dict:
                x_dict[node_type] = encoder(x_dict[node_type])

        # Message passing
        for conv, norm in zip(self.convs, self.norms):
            new_x_dict = conv(x_dict, edge_index_dict)

            updated = {}
            for node_type, x_new in new_x_dict.items():
                x_old = x_dict[node_type]
                x_new = norm[node_type](x_new)

                if x_new.shape == x_old.shape:
                    x_new = x_new + x_old  # residual

                x_new = torch.relu(x_new)
                x_new = nn.functional.dropout(x_new, p=self.dropout, training=self.training)
                updated[node_type] = x_new

            x_dict = updated

        # Edge classification
        edge_preds = {}
        for edge_type, edge_index in edge_index_dict.items():
            src_type, _, dst_type = edge_type
            src, dst = edge_index
            src_x, dst_x = x_dict[src_type], x_dict[dst_type]

            edge_feat = torch.cat([src_x[src], dst_x[dst]], dim=-1)

            if self.use_rel_feats:
                src_feat, dst_feat = data[src_type].x[src], data[dst_type].x[dst]
                delta = src_feat[:, :3] - dst_feat[:, :3]  # Δx, Δy, Δz
                if src_feat.size(1) == 4 and dst_feat.size(1) == 4:
                    dt = src_feat[:, -1] - dst_feat[:, -1]
                    delta = torch.cat([delta, dt.view(-1, 1)], dim=-1)
                edge_feat = torch.cat([edge_feat, delta], dim=-1)

            edge_pred = self.edge_classifiers["__".join(edge_type)](edge_feat).view(-1)
            edge_preds[edge_type] = edge_pred

        return edge_preds


In [6]:
edge_classifier = TrackingHeteroGNN(
    num_layers=5,
    hidden_dim=32,
    dropout=0.1
)
print(f"Number of parameters: {sum(p.numel() for p in edge_classifier.parameters() if p.requires_grad)}")

Number of parameters: 57860


In [7]:
import torch
import torch.nn.functional as F

def hetero_edge_loss(edge_preds, edge_labels, pos_weight_dict=None):
    """
    Compute per-type BCE loss and average across edge types.
    
    pos_weight_dict: optional dict with per-type positive weights 
                     (for class imbalance).
    """
    losses = []
    for edge_type, pred in edge_preds.items():
        target = edge_labels[edge_type].float().to(pred.device)
        pos_weight = None
        if pos_weight_dict and edge_type in pos_weight_dict:
            pos_weight = torch.tensor([pos_weight_dict[edge_type]], device=pred.device)
        loss = F.binary_cross_entropy_with_logits(pred, target, pos_weight=pos_weight)
        losses.append(loss)
    return torch.stack(losses).mean()


In [8]:
total_edges = 0
total_edge_counts = {
    ("pixel", "to", "mppc"): 0,
    ("mppc", "to", "pixel"): 0,
    ("pixel", "to", "pixel"): 0,
    ("mppc", "to", "mppc"): 0
}
positive_edge_counts = {
    ("pixel", "to", "mppc"): 0,
    ("mppc", "to", "pixel"): 0,
    ("pixel", "to", "pixel"): 0,
    ("mppc", "to", "mppc"): 0
}
for data in train_loader:
    for edge_type, labels in data.edge_labels_dict.items():
        total_edges += labels.size(0)
        total_edge_counts[edge_type] += labels.size(0)
        positive_edge_counts[edge_type] += labels.sum().item()

for edge_type in total_edge_counts:
    total = total_edge_counts[edge_type]
    positive = positive_edge_counts[edge_type]
    ratio = (total - positive) / positive if positive > 0 else float('inf')
    print(f"Edge type {edge_type}: {positive}/{total} positive edges, ratio: {ratio:.2f}")
edge_weight_dict = {
    edge_type: ((total_edge_counts[edge_type] - positive_edge_counts[edge_type]) / positive_edge_counts[edge_type])
    for edge_type in total_edge_counts
}

Edge type ('pixel', 'to', 'mppc'): 11680475.0/25124374 positive edges, ratio: 1.15
Edge type ('mppc', 'to', 'pixel'): 11680475.0/25124374 positive edges, ratio: 1.15
Edge type ('pixel', 'to', 'pixel'): 10196314.0/18467104 positive edges, ratio: 0.81
Edge type ('mppc', 'to', 'mppc'): 36260302.0/58232850 positive edges, ratio: 0.61


In [ ]:
import torch
import torch.nn.functional as F
from sklearn.metrics import roc_auc_score
from functools import partial
from tqdm import tqdm


def hetero_edge_loss(edge_preds, edge_labels, pos_weight_dict=None):
    """
    BCE loss averaged across edge types.
    """
    losses = []
    for edge_type, pred in edge_preds.items():
        target = edge_labels[edge_type].float().to(pred.device)
        pos_weight = None
        if pos_weight_dict and edge_type in pos_weight_dict:
            pos_weight = torch.tensor([pos_weight_dict[edge_type]], device=pred.device)
        loss = F.binary_cross_entropy_with_logits(pred, target, pos_weight=pos_weight)
        losses.append(loss)
    return torch.stack(losses).mean()


@torch.no_grad()
def evaluate(model, loader, device="cuda"):
    """
    Evaluate per edge-type: loss, accuracy, AUC.
    """
    model.eval()
    all_metrics = {}
    for hetero_graph in loader:
        hetero_graph = hetero_graph.to(device)
        out = model(hetero_graph)
        edge_labels = hetero_graph.edge_labels_dict
        for edge_type, pred in out.items():
            target = edge_labels[edge_type].float().to(device)

            prob = torch.sigmoid(pred)
            pred_bin = (prob > 0.5).float()

            # compute metrics
            acc = (pred_bin == target).float().mean().item()
            try:
                auc = roc_auc_score(target.cpu().numpy(), prob.cpu().numpy())
            except ValueError:
                auc = float("nan")  # only one class in batch

            if edge_type not in all_metrics:
                all_metrics[edge_type] = {"acc": [], "auc": []}
            all_metrics[edge_type]["acc"].append(acc)
            all_metrics[edge_type]["auc"].append(auc)

    # average across batches
    for edge_type in all_metrics:
        all_metrics[edge_type]["acc"] = sum(all_metrics[edge_type]["acc"]) / len(all_metrics[edge_type]["acc"])
        all_metrics[edge_type]["auc"] = sum(all_metrics[edge_type]["auc"]) / len(all_metrics[edge_type]["auc"])
    return all_metrics


def train(
    model,
    train_loader,
    val_loader=None,
    edge_weight_dict=None,
    num_epochs=20,
    lr=1e-3,
    device="cuda"
):
    """
    End-to-end training loop with validation.
    """
    model = model.to(device)
    criterion = partial(hetero_edge_loss, pos_weight_dict=edge_weight_dict)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)

    for epoch in range(num_epochs):
        model.train()
        total_loss = 0
        for hetero_graph in tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}"):
            hetero_graph = hetero_graph.to(device)

            optimizer.zero_grad()
            out = model(hetero_graph)
            loss = criterion(out, hetero_graph.edge_labels_dict)
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        avg_loss = total_loss / len(train_loader)
        print(f"Epoch {epoch+1}, Train Loss: {avg_loss:.4f}")

        # optional validation
        if val_loader is not None:
            metrics = evaluate(model, val_loader, device)
            print(f"Validation metrics @ epoch {epoch+1}:")
            for etype, vals in metrics.items():
                print(f"  {etype}: acc={vals['acc']:.3f}, auc={vals['auc']:.3f}")


In [ ]:
train(
    edge_classifier,
    train_loader,
    val_loader=test_loader,
    edge_weight_dict=edge_weight_dict,
    num_epochs=20,
    lr=1e-3,
    device="cuda" if torch.cuda.is_available() else "cpu"
)

In [9]:
from src.torch.training import FocalLoss, HeteroLossWrapper
from sklearn.metrics import roc_auc_score
from functools import partial
criterion = partial(hetero_edge_loss, pos_weight_dict=edge_weight_dict)
optimizer = torch.optim.AdamW(edge_classifier.parameters(), lr=0.001)
num_epochs = 20
for epoch in range(num_epochs):
    edge_classifier.train()
    total_loss = 0
    for hetero_graph in tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}"):
        optimizer.zero_grad()
        out = edge_classifier(hetero_graph)
        # Assuming labels are stored in hetero_graph.y
        loss = criterion(out, hetero_graph.edge_labels_dict)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    avg_loss = total_loss / len(train_loader)
    print(f"Epoch {epoch+1}, Loss: {avg_loss:.4f}")

Epoch 1/20: 100%|██████████| 210/210 [01:11<00:00,  2.93it/s]


Epoch 1, Loss: 0.6329


Epoch 2/20: 100%|██████████| 210/210 [01:24<00:00,  2.50it/s]


Epoch 2, Loss: 0.6015


Epoch 3/20: 100%|██████████| 210/210 [01:34<00:00,  2.23it/s]


...

Epoch 18, Loss: 0.5498


Epoch 19/20: 100%|██████████| 210/210 [01:38<00:00,  2.14it/s]


Epoch 19, Loss: 0.5437


Epoch 20/20: 100%|██████████| 210/210 [01:40<00:00,  2.10it/s]

Epoch 20, Loss: 0.5471


In [10]:
predictions = {}
labels = {}
edge_classifier.eval()
with torch.no_grad():
    for hetero_graph in tqdm(test_loader, desc="Evaluating"):
        out = edge_classifier(hetero_graph)
        for edge_type in out:
            if edge_type not in predictions:
                predictions[edge_type] = []
                labels[edge_type] = []
            predictions[edge_type].append(torch.sigmoid(out[edge_type].cpu()))
            labels[edge_type].append(hetero_graph.edge_labels_dict[edge_type].cpu())
    for edge_type in predictions:
        predictions[edge_type] = torch.cat(predictions[edge_type])
        labels[edge_type] = torch.cat(labels[edge_type])
        auc = roc_auc_score(labels[edge_type].numpy(), predictions[edge_type].numpy())
        print(f"Edge type: {edge_type}, AUC: {auc:.4f}")

Evaluating: 100%|██████████| 90/90 [00:20<00:00,  4.49it/s]


Edge type: ('pixel', 'to', 'pixel'), AUC: 0.8173
Edge type: ('mppc', 'to', 'mppc'), AUC: 0.7538
Edge type: ('pixel', 'to', 'mppc'), AUC: 0.7511
Edge type: ('mppc', 'to', 'pixel'), AUC: 0.7499
